# Azure Blob Storage

## 1. What is Azure Blob Storage?

**Azure Blob Storage** is Azure's object storage service for storing unstructured data such as:

- PDFs
- Images
- Videos
- CSV/JSON files
- Office documents
- Logs
- Audio files
- Training datasets

For your **GenAI/RAG architecture**, Blob Storage is commonly used as the **document/data storage layer**.

```text
User
 ↓
Upload Document
 ↓
Azure Blob Storage
 ↓
Document Processing
 ↓
Azure AI Search
 ↓
RAG
```

---

# 2. Why Blob Storage in AI Applications?

Suppose an enterprise has:

```text
HR Policies
Invoices
Contracts
Product Manuals
Insurance Documents
```

Instead of storing these directly in the application server:

```text
❌ Application Server
   ├── PDFs
   ├── DOCX
   └── Images
```

use:

```text
✅ Azure Blob Storage
   ├── hr/
   ├── invoices/
   ├── contracts/
   └── products/
```

The application processes the documents when required.

---

# 3. Blob Storage Hierarchy ⭐⭐⭐⭐⭐

Remember this structure:

```text
Storage Account
      │
      └── Container
             │
             ├── file1.pdf
             ├── file2.pdf
             └── folder/file3.pdf
```

### Storage Account

Top-level Azure storage resource.

```text
Storage Account
```

### Container

Logical grouping of blobs.

```text
Storage Account
 ├── hr-documents
 ├── invoices
 └── contracts
```

### Blob

Individual object/file.

```text
hr-documents
 ├── leave-policy.pdf
 ├── employee-handbook.pdf
 └── benefits.pdf
```

---

# 4. Types of Blobs

Azure Blob Storage supports three blob types:

| Blob Type | Usage |
|---|---|
| **Block Blob** | General files, PDFs, images, documents |
| **Append Blob** | Append-oriented workloads such as logs |
| **Page Blob** | Random-access/page-oriented data, including VHD scenarios |

For most **RAG/document-processing workloads**, you will typically use **Block Blobs**.

---

# 5. Blob Storage + RAG ⭐⭐⭐⭐⭐

This is the most important connection for your interview.

```text
                    DOCUMENT INGESTION

PDF / DOCX
    │
    ▼
Azure Blob Storage
    │
    ▼
Azure Function
    │
    ▼
Document Intelligence
    │
    ▼
Text + Tables + Metadata
    │
    ▼
Chunking
    │
    ▼
Embeddings
    │
    ▼
Azure AI Search
```

Then at query time:

```text
User Question
     ↓
Azure AI Search
     ↓
Relevant Chunks
     ↓
Azure OpenAI
     ↓
Answer
```

Blob Storage is therefore generally the **source/document storage layer**, while Azure AI Search is the **retrieval/indexing layer**.

---

# 6. Blob Trigger + Azure Functions

This is a very common production architecture.

Suppose a user uploads:

```text
leave-policy.pdf
```

```text
User
 ↓
Blob Storage
 ↓
Blob Created Event
 ↓
Azure Function
 ↓
Document Intelligence
 ↓
Chunking
 ↓
Embeddings
 ↓
Azure AI Search
```

This creates an **event-driven RAG ingestion pipeline**.

---

# 7. Why Use Event-Driven Processing?

Suppose the user uploads a 200-page PDF.

Don't do:

```text
HTTP Request
 ↓
Upload
 ↓
Extract 200 pages
 ↓
Chunk
 ↓
Embedding
 ↓
Index
 ↓
Return response
```

The HTTP request could remain open for a long time.

Instead:

```text
Upload
 ↓
Blob Storage
 ↓
Event
 ↓
Queue
 ↓
Background Processing
```

The user can receive:

> "Document uploaded. Processing has started."

while the backend processes the document asynchronously.

---

# 8. Simple Python Upload

Install:

```bash
pip install azure-storage-blob azure-identity
```

Using a connection string is simple for learning:

```python
from azure.storage.blob import BlobServiceClient

connection_string = "YOUR_CONNECTION_STRING"

client = BlobServiceClient.from_connection_string(
    connection_string
)

container = client.get_container_client(
    "documents"
)

with open("hr_policy.pdf", "rb") as file:

    container.upload_blob(
        name="hr_policy.pdf",
        data=file,
        overwrite=True
    )
```

---

# 9. Production Authentication

For production, prefer **Microsoft Entra ID + Managed Identity** where supported rather than embedding storage keys.

```text
Azure Function
      │
Managed Identity
      ↓
Microsoft Entra ID
      ↓
RBAC
      ↓
Blob Storage
```

Python:

```python
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

credential = DefaultAzureCredential()

client = BlobServiceClient(
    account_url="https://mystorage.blob.core.windows.net",
    credential=credential
)
```

This fits directly with the previous topic:

```text
Managed Identity
+
Entra ID
+
RBAC
+
Blob Storage
```

---

# 10. RBAC

Don't give your AI application unrestricted storage access.

For example:

```text
AI Function
     │
Managed Identity
     │
     ▼
Blob Storage
     │
     ▼
Required Storage RBAC Role
```

Follow **least privilege**.

For example, if the application only needs to read documents, don't give it unnecessary write/delete permissions.

---

# 11. Blob Storage + Document Intelligence

For enterprise document processing:

```text
PDF
 ↓
Blob Storage
 ↓
Azure Function
 ↓
Azure Document Intelligence
 ↓
Extract:
 ├── Text
 ├── Tables
 ├── Layout
 └── OCR
 ↓
Chunking
 ↓
Embeddings
 ↓
Azure AI Search
```

This is particularly useful for:

- Scanned PDFs
- Invoices
- Contracts
- Forms
- Tables
- Documents containing images/OCR content

---

# 12. Blob Metadata

You can maintain metadata for documents.

Example:

```text
Blob
 ├── name: leave-policy.pdf
 ├── department: HR
 ├── country: India
 ├── document_type: policy
 └── version: 3
```

Metadata can help with:

- Document classification
- Filtering
- Processing decisions
- Auditability
- RAG source tracking

---

# 13. Blob Storage + Azure AI Search

Important distinction:

```text
Blob Storage
    ↓
Stores original documents

Azure AI Search
    ↓
Indexes searchable content
```

For example:

```text
Blob:
leave-policy.pdf
        │
        ▼
Document extraction
        │
        ▼
Chunks:
 ├── eligibility
 ├── annual leave
 ├── carry-forward
 └── exceptions
        │
        ▼
Azure AI Search
```

When the user asks:

> "How many leave days can I carry forward?"

Azure AI Search retrieves the relevant chunk rather than reading the entire PDF from Blob Storage.

---

# 14. Blob Storage + Agentic AI

An agent can use Blob Storage indirectly through tools.

Example:

> "Summarize the latest contract."

```text
User
 ↓
Agent
 ↓
Document Tool
 ↓
Blob Storage
 ↓
Document Intelligence
 ↓
Extract Content
 ↓
Agent
 ↓
Azure OpenAI
 ↓
Summary
```

Or:

```text
Agent
 ├── Search Tool → Azure AI Search
 ├── Document Tool → Blob Storage
 └── Business Tool → REST API
```

---

# 15. Blob Storage + Azure Functions + Agent

A very common architecture:

```text
                     USER
                       │
                       ▼
                 Upload Document
                       │
                       ▼
                Azure Blob Storage
                       │
                  Blob Event
                       │
                       ▼
                Azure Function
                       │
              ┌────────┴────────┐
              ▼                 ▼
      Document Intelligence   Metadata
              │
              ▼
           Chunking
              │
              ▼
          Embeddings
              │
              ▼
       Azure AI Search
              │
              ▼
             RAG
              │
              ▼
         Azure OpenAI
```

---

# 16. Storage Tiers

Azure Blob Storage provides access tiers such as:

| Tier | Typical Usage |
|---|---|
| **Hot** | Frequently accessed data |
| **Cool** | Infrequently accessed data |
| **Cold** | Rarely accessed data |
| **Archive** | Long-term archival |

For example:

```text
Current HR policies
       ↓
Hot

Old contracts
       ↓
Cool / Cold

Historical documents
       ↓
Archive
```

The appropriate tier depends on access frequency, retrieval requirements, latency requirements, and cost.

---

# 17. Lifecycle Management

You can automatically move data between tiers.

Example:

```text
New document
     ↓
Hot
     ↓
After 30 days
     ↓
Cool
     ↓
After 180 days
     ↓
Cold
     ↓
Long-term
     ↓
Archive
```

This helps optimize storage costs.

---

# 18. Security

For production AI workloads, consider:

```text
Authentication
     ↓
Entra ID
     ↓
Managed Identity
     ↓
RBAC
     ↓
Blob Storage
```

Additional controls can include:

- Encryption at rest
- HTTPS
- Private endpoints
- Network restrictions
- Storage firewall
- Microsoft Defender for Storage
- Access policies
- Versioning
- Soft delete

---

# 19. SAS Tokens ⭐⭐⭐⭐⭐

**Shared Access Signature (SAS)** provides delegated, limited access to storage resources.

Example:

```text
Application
    ↓
Generate SAS
    ↓
Temporary URL
    ↓
Client
    ↓
Blob
```

A SAS can restrict:

- Resource
- Permissions
- Start/end time
- Protocol
- Scope

Example use case:

> Allow a user to upload a PDF directly to Blob Storage without giving the client the storage account key.

```text
Browser
 ↓
SAS URL
 ↓
Blob Storage
```

### Important

SAS is not the same as Managed Identity.

```text
Managed Identity
→ workload authentication

SAS
→ delegated, time-limited access
```

---

# 20. Blob Storage vs Azure Files

Common interview question.

| Blob Storage | Azure Files |
|---|---|
| Object storage | Managed file shares |
| Unstructured data | File-system style access |
| HTTP/REST access | SMB/NFS support depending on configuration |
| PDFs/images/videos | Shared application files |
| Excellent for data lakes/RAG documents | File-share scenarios |

For a RAG document repository:

> **Blob Storage is generally the natural choice.**

---

# 21. Blob Storage vs Database

Don't use Blob Storage as your primary relational database.

```text
Blob Storage
→ Documents / binary objects

Azure SQL
→ Relational structured data

Cosmos DB
→ NoSQL/document data

Azure AI Search
→ Search/index/retrieval
```

In an enterprise RAG application, these can work together:

```text
                 AI Application
                       │
       ┌───────────────┼──────────────┐
       ▼               ▼              ▼
 Blob Storage      Azure SQL      AI Search
 Documents         Business DB    RAG Index
```

---

# 22. Production RAG Architecture

```text
                           USER
                             │
                             ▼
                       Web / Teams
                             │
                             ▼
                    Azure API Management
                             │
                             ▼
                       AI Backend
                             │
                       LangGraph
                             │
              ┌──────────────┼──────────────┐
              │              │              │
              ▼              ▼              ▼
        Azure OpenAI    Azure AI Search   APIs
                             ▲
                             │
                      Indexed Documents
                             ▲
                             │
                    ┌─────────────────┐
                    │ Azure Functions │
                    └────────┬────────┘
                             │
                    Document Intelligence
                             ▲
                             │
                    Azure Blob Storage
                             ▲
                             │
                         Documents
```

Security:

```text
Entra ID
Managed Identity
RBAC
Private Endpoint
Key Vault where required
```

Observability:

```text
Application Insights
Azure Monitor
```

Safety:

```text
Azure AI Content Safety
```

---

# 23. Interview Questions

### Q1. What is Azure Blob Storage?

> "Azure Blob Storage is Azure's object storage service for storing unstructured data such as documents, images, videos and other files."

### Q2. How would you use Blob Storage in a RAG application?

> "I would use Blob Storage as the source repository for enterprise documents. When a document is uploaded, an event can trigger an Azure Function, which extracts the content using Document Intelligence, chunks it, generates embeddings and indexes the chunks into Azure AI Search."

### Q3. Why not store PDFs directly in Azure AI Search?

> "I would generally keep the original documents in Blob Storage and store searchable extracted content, vectors and metadata in Azure AI Search. This separates source storage from retrieval/indexing."

### Q4. How would you process a 200-page PDF?

> "I would upload it to Blob Storage and process it asynchronously using an event-driven pipeline. A Blob event or queue can trigger Azure Functions, which can invoke Document Intelligence, perform chunking and embedding, and index the resulting chunks into Azure AI Search."

### Q5. How do you secure Blob Storage?

> "I would use Microsoft Entra ID and Managed Identity with least-privilege RBAC, and apply network controls such as private endpoints where required. For temporary delegated client access, I could use appropriately scoped SAS tokens."

### Q6. What is a Blob Trigger?

> "It's an Azure Functions trigger that executes processing in response to Blob Storage events, making it useful for event-driven document ingestion."

### Q7. What is SAS?

> "Shared Access Signature provides scoped, time-limited delegated access to Azure Storage resources without exposing the storage account key."

---

# 24. Senior-Level Scenario

### Interviewer:

> "Design a document ingestion system for an enterprise RAG application."

### Strong answer:

> "I would use Azure Blob Storage as the durable source repository. Documents uploaded to a container would generate an event that triggers an asynchronous processing pipeline, typically through Azure Functions and a queue or Service Bus. The processing layer would use Azure Document Intelligence for complex PDFs and OCR, perform structure-aware chunking and metadata extraction, generate embeddings, and index the chunks and vectors into Azure AI Search. I would preserve document ID, page and section metadata for citations and traceability. For security, I would use Managed Identity, Entra ID and least-privilege RBAC. Application Insights and Azure Monitor would provide operational observability."

### Whiteboard:

```text
Document
   │
   ▼
Blob Storage
   │
   ▼
Event / Queue
   │
   ▼
Azure Function
   │
   ▼
Document Intelligence
   │
   ▼
Chunk + Metadata
   │
   ▼
Embeddings
   │
   ▼
Azure AI Search
   │
   ▼
RAG
   │
   ▼
Azure OpenAI
   │
   ▼
Answer + Citations
```

**Key mental model:**

> **Blob Storage = source/document storage → Azure Functions = processing → Document Intelligence = extraction → AI Search = indexing/retrieval → Azure OpenAI = generation.**